---
title: Transformers
---

::: {.callout-tip}
## Slides
[Slides](https://docs.google.com/presentation/d/1ClhiFBnehwgFGDzL6nayEdUijLlvo1W_7KlPy5XO_AI/edit)
::: 


## Architecture Types 

There are three types or architectures of transformer models. These correspond to the attention patterns used within the model and subsequently the tasks they are best suited for.

The three types are: 

1. <mark>Encoder-only</mark>:  An encoder transforms the text embeddings into a representation that can support a variety of tasks. The attention mechanism allows each token to attend to all other tokens in the input sequence. Popular examples include BERT and RoBERTa. 

2. <mark>Decoder-only</mark>:  A decoder predicts the next token to continue the input text. The attention mechanism is typically causal, meaning each token can only attend to previous tokens in the sequence. Popular examples include GPT and Llama.

3. <mark>Encoder-decoder</mark>:  Encoder-decoders are used in sequence-to-sequence tasks, where one text string is converted into another (e.g., machine translation). The encoder uses bidirectional self-attention, while the decoder uses causal self-attention and cross-attention to the encoder outputs. Popular examples include T5 and BART.

<img src="https://heidloff.net/assets/img/2023/02/transformers.png" width="100%" style="filter: invert(1);"> 

| Architecture | Attention pattern | Example task | 
|:---|:---|:---|
| Encoder-only | Usually bidirectional self-attention | Classification or masked-token prediction |
| Decoder-only | Causal self-attention | Text generation |
| Encoder–decoder | Encoder self-attention, causal decoder self-attention, and cross-attention | Translation |

<br/>

## Training Steps 

Training a transformer model typically involves the following steps:

1. Pre-training

The model is initially trained on a large corpus of text using self-supervised objectives, such as predicting masked tokens or the next token in a sequence.

2. Post-training: Supervised Fine-tuning (SFT)

After pre-training, the model is fine-tuned on a smaller, task-specific dataset with supervised learning to adapt it to particular tasks.

3. Post-training: Reinforcement Learning with Human Feedback (RLHF)

In reinforcement learning, the model is further trained using feedback from its performance on specific tasks. This can involve optimizing a reward signal that reflects the quality of the model's outputs, allowing it to improve beyond what is possible with supervised fine-tuning alone.

| | Encoder-only 	| Decoder-only | Encoder-decoder |
|---|---|---|---|
| Pre-training 	| Masked LM: predict ~15% masked tokens from both sides	| Next-token prediction on raw text	| Denoising: corrupt the input.<br/>Encoder reads the corrupted text, decoder rebuilds the missing parts |
| Supervised Fine-Tuning	| Task fine-tuning: add a head and train on labeled data e.g. classification, NER, QA span extraction 	| Instruction tuning: prompt → response pairs in one sequence | Text-to-text: prompt goes to the encoder, response to the decoder  |
| Reinforcement Learning with Human Feedback 	| N/A	| Standard pipeline: train a reward model then optimize with PPO plus a KL penalty to the SFT model	| Works in principle: the decoder is the policy, and rewards and log-probs are computed over decoder outputs given the encoder input. Rarely done at scale |

## Supporting Components of Transformers

The main computational components of transformers are the attention mechanism and the feed-forward network.

Transformers often consist of a multi-head self-attention unit (which allows the word representations to interact with each other) followed by a fully connected network $\textbf{mlp}[\mathbf{x}_\bullet]$ (that operates separately on each
word). 

In addition, there are other units that are typically included in a transformer. These include 

1. Positional encodings (to provide information about the order of tokens in the sequence)
2. Layer normalization (LayerNorm)
3. Residual connections


## Positional encoding

Representing each input $\mathbf{x}_n$ by its content alone does not explicitly encode its position in a sequence. For language, however, order is important. The sentence

> The woman ate the raccoon.

has a different meaning from

> The raccoon ate the woman.

Positional encodings provide information about where inputs occur in a sequence. There are two main approaches: **absolute positional encodings** and **relative positional encodings**.

---

<img src="assets/Chap12/TransformerPE.svg" style="filter: invert(1);" width="60%" align="right">

**Positional encodings.** A positional encoding matrix $\Pi$ can be added to the data matrix $\mathbf{X}$ to give each input information about its position. Each column of $\Pi$ represents a different position. In this example, the encodings follow a predefined sinusoidal pattern:

---

**Absolute positional encodings**

Absolute positional encodings identify each input’s position in the sequence. A matrix $\Pi$ with the same dimensions as $\mathbf{X}$ is added to the input representations:

$$
\widetilde{\mathbf{X}} = \mathbf{X} + \Pi.
$$

$$
\Pi_{d,n} = \begin{cases}
\sin\left(\frac{n}{10000^{d/D}}\right) & \text{if } d \text{ is even}, \\
\cos\left(\frac{n}{10000^{(d-1)/D}}\right) & \text{if } d \text{ is odd},
\end{cases}
$$

where $n$ indexes positions, $d=0,\ldots,D-1$ indexes features, and $D$ is the encoding dimension. This formula can generate encodings for additional positions if needed. Alternatively, positional encodings can be learned during training.

Each column of $\Pi$ provides a distinct vector for the corresponding position. Thus, the representation of a word depends on both its content and its location in the sequence.

The positional vectors can be predefined, as in the sinusoidal construction above, or learned during training. They can be introduced at the network input or incorporated at multiple layers.

**Relative positional encodings**

Relative positional encodings describe where one input occurs in relation to another. For example, the relationship “two words before” remains the same whether the words appear near the beginning or the end of a sentence.

For inputs at positions $a$ and $b$, their relative position can be represented by the signed offset

$$
r = a-b.
$$

The sign indicates whether position $a$ comes before or after position $b$, and the magnitude indicates the distance between them.

Although this offset can be computed from absolute positions, relative positional encodings represent it directly. One approach assigns a learned parameter or vector $\pi_r$ to each offset $r$. The network uses this encoding when combining information from the two positions. Pairs of inputs with the same offset share the same relative positional encoding, regardless of their absolute locations.

---


One thing that’s missing from the model as we have described it so far is a way to account for the order of the words in the input sequence.

To address this, the transformer adds a vector to each input embedding. These vectors follow a specific pattern that the model learns, which helps it determine the position of each word, or the distance between different words in the sequence. The intuition here is that adding these values to the embeddings provides meaningful distances between the embedding vectors once they’re projected into Q/K/V vectors and during dot-product attention.

<img src="https://jalammar.github.io/images/t/transformer_positional_encoding_vectors.png" style="filter: invert(1);" width="100%">

To give the model a sense of the order of the words, we add positional encoding vectors -- the values of which follow a specific pattern.

If we assumed the embedding has a dimensionality of 4, the actual positional encodings would look like this:

<img src="https://jalammar.github.io/images/t/transformer_positional_encoding_example.png" style="filter: invert(1);" width="100%">

A real example of positional encoding with a toy embedding size of 4

What might this pattern look like?

In the following figure, each row corresponds to a positional encoding of a vector. So the first row would be the vector we’d add to the embedding of the first word in an input sequence. Each row contains 512 values – each with a value between 1 and -1. We’ve color-coded them so the pattern is visible.

<img src="https://jalammar.github.io/images/t/transformer_positional_encoding_large_example.png" style="filter: invert(1);" width="100%">

A real example of positional encoding for 20 words (rows) with an embedding size of 512 (columns). You can see that it appears split in half down the center. That's because the values of the left half are generated by one function (which uses sine), and the right half is generated by another function (which uses cosine). They're then concatenated to form each of the positional encoding vectors.

The formula for positional encoding is described in the paper (section 3.5). You can see the code for generating positional encodings in [`get_timing_signal_1d()`](https://github.com/tensorflow/tensor2tensor/blob/23bd23b9830059fbc349381b70d9429b5c40a139/tensor2tensor/layers/common_attention.py). This is not the only possible method for positional encoding. It, however, gives the advantage of being able to scale to unseen lengths of sequences (e.g. if our trained model is asked to translate a sentence longer than any of those in our training set).

**July 2020 Update:** 
The positional encoding shown above is from the Tensor2Tensor implementation of the Transformer. The method shown in the paper is slightly different in that it doesn’t directly concatenate, but interweaves the two signals. The following figure shows what that looks like. [Here’s the code to generate it](https://github.com/jalammar/jalammar.github.io/blob/master/notebookes/transformer/transformer_positional_encoding_graph.ipynb):

<img src="https://jalammar.github.io/images/t/attention-is-all-you-need-positional-encoding.png" style="filter: invert(1);" width="100%">



## Layer Normalization

Suppose a token has a four-dimensional embedding:

$$ x=[2,4,6,8] $$

Its mean is:
  
$$ \mu=\frac{2+4+6+8}{4}=5 $$

Its standard deviation is:

$$ \sigma=\sqrt{\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}}=\sqrt{5} $$

LayerNorm first standardizes the vector:

$$ \hat{x}=\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}} $$

Ignoring the tiny \\(\epsilon\\) for simplicity:

$$ \hat{x}\approx[-1.34,-0.45,0.45,1.34] $$

Now the vector has approximately:

- Mean = 0
- Standard deviation = 1

But LayerNorm doesn't stop there. It also learns a scale and shift for each dimension:

$$ \boxed{y=\gamma\odot\hat{x}+\beta} $$

Here, $\gamma$ and $\beta$ are learnable parameters, each with four elements in this example. They allow the network to adjust the standardized representation.

<b><u>What exactly gets normalized in a transformer?</u></b>

Suppose the input has shape:

$$ (\text{batch},\text{tokens},\text{embedding dimensions}) =(32,64,128) $$

LayerNorm normalizes each token's 128-dimensional vector independently.

LayerNorm operates across embedding dimensions

Each row is normalized separately. The same learned γ and β parameters are shared across token positions.

This is different from Batch Normalization, which uses statistics across examples in a batch.

**<u>Where does LayerNorm appear in our transformer?</u>**

Recall our implementation:

```
attention_output = self.attention(x)
x = self.norm1(x + attention_output)
ff_output = self.feedforward(x)
x = self.norm2(x + ff_output)
```

The first LayerNorm normalizes the result of the attention block plus its residual connection.

The second normalizes the result of the feed-forward block plus its residual connection.

This is called a Post-LayerNorm transformer, because normalization happens after each residual addition. Many modern transformer architectures instead use Pre-LayerNorm, which normalizes before each sublayer.

**<u>PyTorch implementation</u>**

```
import torch
import torch.nn as nn
x = torch.tensor([[2., 4., 6., 8.]])
norm = nn.LayerNorm(4)
y = norm(x)
print(y)
```

The important detail is that `nn.LayerNorm(4)` creates 8 learnable parameters:

- Four scale parameters (\\(\gamma\\))
- Four shift parameters (\\(\beta\\))

For our transformer with `d_model=128`, each LayerNorm has 256 learnable parameters.

**<u>Why do we need it?</u>**

During training, the outputs of attention and feed-forward layers can change in magnitude as their parameters are updated. LayerNorm helps keep the scale of representations more controlled, making optimization more stable.

It also helps gradients propagate through deep transformer networks.

One important distinction: LayerNorm does not make all token embeddings identical or erase their contextual information. It normalizes each vector's mean and variance, while preserving differences in the relative values of its dimensions.

In short, self-attention determines what information each token receives; LayerNorm helps keep the resulting representations numerically well-behaved.

## Residual connections

Residual connections are a key component of transformer layers. They allow the network to retain information from earlier layers by adding the input of a layer to its output. This helps mitigate the vanishing gradient problem and enables the training of very deep networks.

Mathematically, if a layer implements a function $F(\mathbf{X})$, the residual connection produces an output $\mathbf{X} + F(\mathbf{X})$. In transformers, both the multi-head self-attention and the feed-forward sub-layers use residual connections.

Transformers are a type of neural network architecture designed to handle sequential data, such as text, by leveraging self-attention mechanisms to capture dependencies between tokens regardless of their distance in the sequence. They were introduced in the paper <a href='https://arxiv.org/abs/1706.03762'><mark>Attention is All You Need</mark></a> by Vaswani et al. in 2017. 

The paper had the following figure on the overall architecture of the transformer model:

<img src="https://arxiv.org/html/1706.03762v7/Figures/ModalNet-21.png" width="60%" align="right" style="filter:invert(1);">

Note that we have have already studied the feed forward layers in the figure and are familiar with Linear and Softmax. The modules from the figure that we have not yet covered include the Multi-Head Attention, the Masked Multi-Head Attention and Add & Norm layers.

Transformers were initially targeted at natural language processing (NLP) problems. The model's <mark>input is a series of high-dimensional embeddings representing words</mark> or word fragments aka tokens. Text is split into tokens using a sub-word tokenizer such as BPE (Byte Pair Encoding).

The embeddings of tokens are passed through a series of transformer layers. 

<mark>Natural language datasets have a unique complication that text sequences vary in length, and unlike images, there is no easy way to resize them</mark>.
